## **주식 종료 가격 예측 경진대회**

###**1. 데이터**

stock_list.csv: 국내 상장 종목 리스트

* 종목명: 주식 종목 명
* 종목 코드: 주식 종목 코드번호
* 상장시장 : 주식 종목 상장 시장 (KOSPI or KOSDAQ)

* kospi, kosdaq, dJi (다우지수), exc (원/달러 환율) 의 Open(시가) / High(고가) / Volume(거래량) / Change(변동률)

###**2. 코드 흐름**

**1. 라이브러리 설정 및 데이터 로드**

* 데이터 핸들링을 위한 Pandas, NumPy 및 주가/지수 수집을 위한 FinanceDataReader 로드
* 모델링을 위해 XGBoost 구축 및 진행 상황 확인을 위한 tqdm 설정

**2. 경제 지표 수집 및 결측치, 시차 전처리**

* 영업일(Business_days) 기준으로 데이터 프레임에 코스피, 코스닥, 다우지수, 원/달러 환율 데이터 병합
* 미국 휴장일은 전일 데이터로 채워줌
* 한국 휴장일은 data 에서 삭제
* 미국 증시는 시차로 인해 당일 정보를 알 수 없으므로 shift(1)로 1일 밀어주고, 일주일 뒤 종가 예측을 위해 전체 피처 5일씩 이동
* 요일(weekday), 주차(weeknum) 변수를 get_dummies 로 변환 (원-핫 인코딩)

**3. 개별 종목별 파생변수 생성**

* 이동평균선(5일, 10일, 20일) 및 과거 종가/거래량 대비 변화율 파생변수 생성

**4. 모델링**

* 상장 시장에 따라 불필요한 타 시장 지수 컬럼 제거 후 예측일 이전 데이터로 XGBRegressor 학습 및 예측 수행

**5. 성능 평가**

* 실제 종가(true_close) 데이터와 예측값 비교

###**3. 주요 코드 분석**

In [ ]:
for code in tqdm(stock_list['종목코드'].values):
    for day in day_list:
        code_data = fdr.DataReader(code, start = start_date, end = end_date).reset_index()
        code_data['y_Close'] = code_data['Close']

        code_data[[
            'Open', 'High', 'Low', 'Close', 'Volume', 'Change']] = code_data[[
            'Open', 'High', 'Low', 'Close', 'Volume', 'Change']].shift(5)

        #이동 평균선 추가 (5/10/20)
        code_data['5_close'] = code_data['Close'].rolling(window=5).mean()
        code_data['10_close'] = code_data['Close'].rolling(window=10).mean()
        code_data['20_close'] = code_data['Close'].rolling(window=20).mean()

        #6일전/7일전 종가 추가
        code_data['6d_Close'] = code_data[['Close']].shift(1)
        code_data['7d_Close'] = code_data[['6d_Close']].shift(1)

        #6일전/7일전 종가와 전일 종가의 차이 비율 추가
        code_data['diff_Close'] = (code_data['Close'] - code_data['6d_Close'])/code_data['Close']
        code_data['diff_Close2'] = (code_data['Close'] - code_data['7d_Close'])/code_data['Close']

        #6일전 거래량 추가
        code_data['6d_Volume'] = code_data[['Volume']].shift(6)

        #2일전 거래량과 전일 거래량의 차이 추가
        code_data['diff_Volume'] = code_data['Volume'] - code_data['6d_Volume']

        df = pd.merge(m_data, code_data, how='left', on='Date')
        df = df.drop(df[df.isnull().any(axis=1)].index, axis=0)

        market = stock_list[stock_list['종목코드'].str.contains(code)]['상장시장'].values[0]

        if market == 'KOSPI':
            df_kospi = df.drop(['kosdaq_Close', 'kosdaq_Open', 'kosdaq_High', 'kosdaq_Low', 'kosdaq_Volume', 'kosdaq_Change'], axis=1)
            x_train = df_kospi[df_kospi['Date'] < day].drop(['Date','y_Close'], axis=1)
            y_train = df_kospi[df_kospi['Date'] < day]['y_Close']
            xgb.fit(x_train, y_train)
            pred = xgb.predict(df_kospi[df_kospi['Date']==day].drop(['Date','y_Close'], axis=1))
        else:
            df_kosdaq = df.drop(['kospi_Close', 'kospi_Open', 'kospi_High', 'kospi_Low','kospi_Volume', 'kospi_Change', 'kosdaq_Close', 'kosdaq_Open'], axis=1)
            x_train = df_kosdaq[df_kosdaq['Date'] < day].drop(['Date','y_Close'], axis=1)
            y_train = df_kosdaq[df_kosdaq['Date'] < day]['y_Close']
            xgb.fit(x_train, y_train)
            pred = xgb.predict(df_kosdaq[df_kosdaq['Date']==day].drop(['Date','y_Close'], axis=1))
        submission.loc[day,code] = pred

**시계열 시차 설계**

미래 시점의 데이터를 미리 참조하는 데이터 누수를 막기 위해 y_Close는 그대로 두고 모든 입력 피처를 shift(5)로 5 영업일씩 뒤로 밀어줌

**추세 및 변동성 파생변수 생성**

주가 흐름을 반영하기 위해 5일/10일/20일 이동평균선(rolling().mean())을 추가하고, 직전일 대비 종가 변화율(diff_Close)과 거래량 차이(diff_Volume)를 계산해 m_data와 병합

**상장 시장별 맞춤형 피처 분리 및 학습**

상장 시장별 불필요한 컬럼을 제거한 뒤 예측 기준일(day) 이전 데이터만으로 XGBRegressor 를 학습시켜 당일 종가 예측

###**4. 새롭게 알게 된 점/아쉬운 점**

**1. 새롭게 알게 된 점**

* 한국과 미국의 시차 차이를 반영해 미국 다우지수를 한 번 더 밀어준 전처리 방법이 인상적이다. 시계열 주가 예측에서는 shift() 함수를 정교하게 설계하는 것이 핵심임을 배웠다.

* 환율, 해외 지수 등의 지표와 이동평균선/변화율 같은 파생변수를 결합하면 트리 기반 모델로도 준수한 시계열 예측 성능을 낼 수 있다는 것을 알게 되었다.

**2. 아쉬운 점**

* 정의해 둔 옵션/선물 만기일 리스트가 실제 학습 피처에는 반영되지 않은 점이 아쉽다. 추가했다면 변동성 예측에 도움이 되었을 것이다.

* XGBRegressor()를 별도의 하이퍼파라미터 튜닝이나 검증 없이 기본값 그대로 사용한 점이 아쉽다.
